# Strings, Arrays & Complex Types




Cleaning, splitting, and exploding messy text into usable data

In the past few days, we’ve worked with numbers and dates. That felt neat, right? But let’s be honest: most of the data you’ll deal with in real pipelines isn’t neat. It comes as free text. Product descriptions, customer comments, log lines, CSV fields stuffed inside a single cell — all of these are strings, and they don’t fit cleanly into rows and columns.

If strings are the “wild animals” of data, then arrays and complex types are the enclosures we build to manage them. Spark is particularly strong here, because it treats complex types like first-class citizens. You don’t have to flatten everything into plain strings — you can store arrays, maps, and structs directly in a DataFrame and query them like tables.

Let’s start with strings, then move to arrays, and finally explore how complex types make Spark shine.

### Strings: Cleaning and Transforming
Suppose we have a dataset of product feedback:

product,feedback

PhoneX,"Excellent battery life!"

PhoneY,"Too heavy "

PhoneZ," NA"

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, upper, length, regexp_replace

spark = SparkSession.builder.appName("day13-strings").getOrCreate()

#data as a list of class
data = [("PhoneX", "Excellent battery life!"),
        ("PhoneY", "Too heavy "),
        ("PhoneZ", " NA")]

print(type(data))

df = spark.createDataFrame(data, ["product", "feedback"])
df.show(truncate=False)

Trimming and Case Normalization

In [0]:
df_clean = (
    df.withColumn("feedback_clean", trim(col("feedback")))
      .withColumn("feedback_lower", lower(col("feedback")))
      .withColumn("feedback_upper", upper(col("feedback")))
)

df_clean.show(truncate=False)

- trim removes stray spaces.
- lower / upper standardize case, useful for comparisons.

### String Length and Replace

In [0]:
df_len = (
    df_clean.withColumn("length", length(col("feedback_clean")))
             .withColumn("feedback_no_exclaim", regexp_replace(col("feedback_clean"), "!", ""))
)

df_len.show(truncate=False)

Now you’ve extracted both structural features (length) and cleaned text (removed punctuation).

### Arrays: Splitting Strings into Lists
Sometimes a single string contains multiple values, like a comma-separated list of tags:

product,tags
PhoneX,"fast, sleek, modern"

PhoneY,"heavy, durable"

PhoneZ,"budget, compact, reliable"

In [0]:
from pyspark.sql.functions import split

data_tags = [
    ("PhoneX", "fast, sleek, modern"),
    ("PhoneY", "heavy, durable"),
    ("PhoneZ", "budget, compact, reliable")
]

df_tags = spark.createDataFrame(data_tags, ["product", "tags"])

df_array = df_tags.withColumn("tags_array", split(col("tags"), ", "))
df_array.show(truncate=False)

Now tags_array is an array column, not just text.

### Exploding Arrays
Arrays are great, but sometimes you want one row per element. That’s where explode comes in.

In [0]:
from pyspark.sql.functions import explode

df_exploded = df_array.withColumn("tag", explode(col("tags_array")))
df_exploded.show()

Suddenly, you can aggregate by tags across products. That’s powerful: “Which features are mentioned most often in reviews?”

### Complex Types: Structs and Maps
Spark doesn’t limit you to strings and arrays. It lets you store structs (nested records) and maps (key-value pairs). These come up often when working with JSON or semi-structured data.

### Struct Example
Suppose we have sales data with nested customer info:

In [0]:
from pyspark.sql.functions import struct

data_struct = [
    ("Order1", "Alice", "North", 250),
    ("Order2", "Bob", "South", 450)
]

df_struct = spark.createDataFrame(data_struct, ["order_id", "cust_name", "region", "amount"])

df_nested = df_struct.withColumn("customer", struct(col("cust_name"), col("region")))
df_nested.select("order_id", "customer", "amount").show(truncate=False)

A struct is like a mini-row inside a row.

### Map Example
Maps are useful for key-value attributes, such as specifications:

In [0]:
from pyspark.sql.functions import create_map, lit

df_map = df_struct.withColumn("specs", create_map(
    lit("currency"), lit("USD"),
    lit("payment_mode"), lit("CreditCard")
))

df_map.select("order_id", "specs").show(truncate=False)

Maps let you store flexible dictionaries inside your DataFrame, which is great when your schema isn’t fixed.

### Why Complex Types Matter

Strings are messy but familiar. Arrays, structs, and maps may feel abstract at first, but they’re the bridge between raw text and structured analytics. Instead of forcing everything into flat tables, Spark encourages you to keep structure where it exists.

- Arrays help you represent “one-to-many” relationships (like tags or transactions).
- Structs let you group related fields together (customer info as a block).
- Maps allow flexible attributes without changing schema constantly.

And the beauty is that you can query them naturally, using Spark SQL.

# Wrapping Up
Today you stepped into Spark’s world of strings and complex types:

- Cleaned and normalized text with trimming, casing, regex replace.
- Split messy strings into arrays and exploded them into rows.
- Built structs and maps to represent nested and flexible data.

Why is this important? Because in real pipelines, not everything comes neatly as numbers or dates. A log file, a JSON API, or a product feedback column will challenge you with messy text and irregular structures. Now you have the tools to tame them.

That’s Day 13. You’re no longer afraid of messy strings or nested data.

Next up, Day 14: UDFs and Pandas UDFs — where we’ll learn how to extend Spark with your own Python logic, and when to be careful about performance.